ImportError: cannot import name 'train_and_forecast' from 'models.forecast' (c:\xampp\htdocs\uganda-coffee-platform\notebooks\../src\models\forecast.py)

In [ ]:
import pandas as pd
import plotly.express as px

# Load the World Bank Pink Sheet
df = pd.read_excel(
    "../data/raw/coffee_prices.xlsx",
    sheet_name="Monthly Prices",
    skiprows=4
)

# Keep only what we need
df = df[["Unnamed: 0", "Coffee, Arabica", "Coffee, Robusta"]].copy()
df.columns = ["date_raw", "arabica_usd", "robusta_usd"]

# Drop rows with no date
df = df.dropna(subset=["date_raw"])

# Convert date format from "2025M01" to datetime
df["date"] = pd.to_datetime(df["date_raw"].astype(str).str.replace("M", "-"), format="%Y-%m")

# Drop rows where both prices are missing
df = df.dropna(subset=["arabica_usd", "robusta_usd"], how="all")

# Sort oldest to newest
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
print(df.head())
print(df.tail())
print(df.isnull().sum())

(796, 4)
  date_raw arabica_usd robusta_usd       date
0  1960M01      0.9409    0.696864 1960-01-01
1  1960M02      0.9469    0.688707 1960-02-01
2  1960M03      0.9281    0.688707 1960-03-01
3  1960M04      0.9303    0.684519 1960-04-01
4  1960M05        0.92    0.690692 1960-05-01
    date_raw arabica_usd robusta_usd       date
791  2025M12    8.404673    4.201344 2025-12-01
792  2026M01    8.023494    4.244334 2026-01-01
793  2026M02    7.084546    3.962364 2026-02-01
794  2026M03    7.370927    3.897107 2026-03-01
795  2026M04    7.302142    3.629907 2026-04-01
date_raw       0
arabica_usd    0
robusta_usd    0
date           0
dtype: int64


In [ ]:
print(df.columns.tolist())

['date_raw', 'arabica_usd', 'robusta_usd', 'date']


In [ ]:
# Save cleaned data to processed folder
df[["date", "arabica_usd", "robusta_usd"]].to_csv(
    "../data/processed/coffee_prices_clean.csv",
    index=False
)

print("Saved successfully")

Saved successfully


In [ ]:
# Add 30-day rolling average (monthly data so 3-month rolling)
df["arabica_ma3"] = df["arabica_usd"].rolling(3).mean()
df["robusta_ma3"] = df["robusta_usd"].rolling(3).mean()

# Plot
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"], y=df["arabica_usd"],
    name="Arabica", line=dict(color="#8B4513", width=1.5)
))
fig.add_trace(go.Scatter(
    x=df["date"], y=df["robusta_usd"],
    name="Robusta", line=dict(color="#D2691E", width=1.5)
))
fig.add_trace(go.Scatter(
    x=df["date"], y=df["arabica_ma3"],
    name="Arabica 3M MA", line=dict(color="#8B4513", dash="dot", width=1)
))
fig.add_trace(go.Scatter(
    x=df["date"], y=df["robusta_ma3"],
    name="Robusta 3M MA", line=dict(color="#D2691E", dash="dot", width=1)
))

fig.update_layout(
    title="Uganda Coffee Prices — Arabica vs Robusta (1960–2026)",
    xaxis_title="Date",
    yaxis_title="Price (USD/kg)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [ ]:
# Calculate monthly percentage change and rolling volatility
df["arabica_pct_change"] = df["arabica_usd"].pct_change() * 100
df["robusta_pct_change"] = df["robusta_usd"].pct_change() * 100

# 12-month rolling volatility (standard deviation of monthly returns)
df["arabica_volatility"] = df["arabica_pct_change"].rolling(12).std()
df["robusta_volatility"] = df["robusta_pct_change"].rolling(12).std()

# Plot volatility
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df["date"], y=df["arabica_volatility"],
    name="Arabica volatility", line=dict(color="#8B4513", width=1.5)
))
fig2.add_trace(go.Scatter(
    x=df["date"], y=df["robusta_volatility"],
    name="Robusta volatility", line=dict(color="#D2691E", width=1.5)
))

fig2.update_layout(
    title="Coffee Price Volatility — 12-Month Rolling Std Dev of Monthly Returns",
    xaxis_title="Date",
    yaxis_title="Volatility (%)",
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()

In [ ]:
import sys
sys.path.append("../src")

from processing.clean import load_coffee_prices, add_features

df_test = load_coffee_prices("../data/raw/coffee_prices.xlsx")
df_test = add_features(df_test)

print(df_test.shape)
print(df_test.columns.tolist())

(796, 9)
['date', 'arabica_usd', 'robusta_usd', 'arabica_ma3', 'robusta_ma3', 'arabica_pct_change', 'robusta_pct_change', 'arabica_volatility', 'robusta_volatility']


In [ ]:
from models.risk import risk_summary

arabica_risk = risk_summary(df_test, "arabica")
robusta_risk = risk_summary(df_test, "robusta")

for k, v in arabica_risk.items():
    print(f"{k}: {v}")

print()

for k, v in robusta_risk.items():
    print(f"{k}: {v}")

coffee_type: Arabica
current_price_usd: 7.3021
current_volatility_pct: 7.65
var_95_pct: -9.69
max_drawdown_pct: -83.34
risk_level: Moderate

coffee_type: Robusta
current_price_usd: 3.6299
current_volatility_pct: 9.85
var_95_pct: -9.0
max_drawdown_pct: -92.69
risk_level: Moderate


In [ ]:
from models.simulator import simulate_revenue, run_scenarios

# Use current Arabica price, 1000 tonnes, typical UGX rate
result = simulate_revenue(
    price_usd_per_kg=7.30,
    volume_tonnes=1000,
    ugx_per_usd=3700
)
print("Single simulation:")
for k, v in result.items():
    print(f"  {k}: {v:,.2f}")

print("\nScenario analysis:")
scenarios = run_scenarios(7.30, 1000, 3700)
for name, data in scenarios.items():
    print(f"  {name}: UGX {data['revenue_ugx']:,.0f} ({data['pct_change']}%)")

Single simulation:
  revenue_usd: 7,300,000.00
  revenue_ugx: 27,010,000,000.00
  revenue_per_tonne_ugx: 27,010,000.00

Scenario analysis:
  Baseline: UGX 27,010,000,000 (0.0%)
  UGX weakens 5%: UGX 25,659,500,000 (-5.0%)
  Price drops 10%: UGX 24,309,000,000 (-10.0%)
  Volume falls 15%: UGX 22,958,500,000 (-15.0%)
  Combined stress: UGX 19,629,517,500 (-27.32%)


In [ ]:
from models.forecast import train_and_forecast

print("Training forecast model — this takes about 30 seconds...")
forecast, metrics = train_and_forecast(df_test, "arabica", periods=12)

print("\nModel error metrics:")
print(f"  MAE:  ${metrics['mae']} per kg")
print(f"  RMSE: ${metrics['rmse']} per kg")
print(f"  MAPE: {metrics['mape']}%")

print("\nNext 12 months forecast:")
future_only = forecast[forecast["ds"] > df_test["date"].max()][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].head(12)
print(future_only.to_string(index=False))

ImportError: cannot import name 'train_and_forecast' from 'models.forecast' (c:\xampp\htdocs\uganda-coffee-platform\notebooks\../src\models\forecast.py)

In [16]:
import inspect
import models.forecast
print(inspect.getsource(models.forecast))

import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


def prepare_prophet_df(df: pd.DataFrame, coffee_type: str = "arabica") -> pd.DataFrame:
    """Prophet requires columns named exactly 'ds' and 'y'."""
    col = f"{coffee_type}_usd"
    prophet_df = df[["date", col]].copy()
    prophet_df.columns = ["ds", "y"]
    prophet_df = prophet_df.dropna()
    return prophet_df


def train_and_forecast(
    df: pd.DataFrame,
    coffee_type: str = "arabica",
    periods: int = 12
) -> tuple[pd.DataFrame, dict]:
    """
    Train a Prophet model and forecast future prices.
    periods: number of months to forecast ahead
    Returns forecast dataframe and error metrics dict.
    """
    prophet_df = prepare_prophet_df(df, coffee_type)

    # Hold out last 12 months for evaluation
    train = prophet_df.iloc[:-12]
    test = prophet_df.iloc[-12:]

    model = Prophet(
        interval_width=0.95,
        yearly_se

In [17]:
import importlib
import sys

# Remove cached version completely
if "models.forecast" in sys.modules:
    del sys.modules["models.forecast"]
if "models" in sys.modules:
    del sys.modules["models"]

# Re-import fresh
from models.forecast import train_and_forecast
print("Import successful")

SyntaxError: invalid syntax (forecast.py, line 71)

In [1]:
import sys
if "models.forecast" in sys.modules:
    del sys.modules["models.forecast"]
if "models" in sys.modules:
    del sys.modules["models"]

from models.forecast import train_and_forecast
print("Import successful")

ModuleNotFoundError: No module named 'models'

In [2]:
import sys
if "models.forecast" in sys.modules:
    del sys.modules["models.forecast"]
if "models" in sys.modules:
    del sys.modules["models"]

from models.forecast import train_and_forecast
print("Import successful")

ModuleNotFoundError: No module named 'models'

In [3]:
import sys
sys.path.append("../src")


In [4]:
from models.forecast import train_and_forecast
print("Import successful")

Import successful


In [5]:
print("Training forecast model — this takes about 30 seconds...")
forecast, metrics = train_and_forecast(df_test, "arabica", periods=12)

print("\nModel error metrics:")
print(f"  MAE:  ${metrics['mae']} per kg")
print(f"  RMSE: ${metrics['rmse']} per kg")
print(f"  MAPE: {metrics['mape']}%")

print("\nNext 12 months forecast:")
future_only = forecast[forecast["ds"] > df_test["date"].max()][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].head(12)
print(future_only.to_string(index=False))

Training forecast model — this takes about 30 seconds...


NameError: name 'df_test' is not defined

In [6]:
import sys
sys.path.append("../src")

from processing.clean import load_coffee_prices, add_features
from models.forecast import train_and_forecast

df_test = load_coffee_prices("../data/raw/coffee_prices.xlsx")
df_test = add_features(df_test)

print("Data loaded:", df_test.shape)

Data loaded: (796, 9)


In [7]:
print("Training forecast model — this takes about 30 seconds...")
forecast, metrics = train_and_forecast(df_test, "arabica", periods=12)

print("\nModel error metrics:")
print(f"  MAE:  ${metrics['mae']} per kg")
print(f"  RMSE: ${metrics['rmse']} per kg")
print(f"  MAPE: {metrics['mape']}%")

print("\nNext 12 months forecast:")
future_only = forecast[forecast["ds"] > df_test["date"].max()][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].head(12)
print(future_only.to_string(index=False))

Training forecast model — this takes about 30 seconds...


22:01:01 - cmdstanpy - INFO - Chain [1] start processing
22:01:02 - cmdstanpy - INFO - Chain [1] done processing



Model error metrics:
  MAE:  $3.0033 per kg
  RMSE: $3.1073 per kg
  MAPE: 36.58%

Next 12 months forecast:
        ds     yhat  yhat_lower  yhat_upper
2026-05-01 5.187791    3.481019    7.166936
2026-06-01 5.096273    3.238626    6.944020
2026-07-01 5.025236    3.113889    6.915082
2026-08-01 5.043554    3.152899    6.987861
2026-09-01 5.089816    3.276875    7.031075
2026-10-01 5.059579    3.230923    6.875248
2026-11-01 5.107300    3.154984    7.068302
2026-12-01 5.192187    3.442641    7.020123
2027-01-01 5.224874    3.395035    6.984234
2027-02-01 5.350910    3.432627    7.146319
2027-03-01 5.550498    3.566877    7.396347
2027-04-01 5.485822    3.681323    7.304406


In [9]:
import plotly.graph_objects as go

fig = go.Figure()

# Historical prices
fig.add_trace(go.Scatter(
    x=df_test["date"],
    y=df_test["arabica_usd"],
    name="Historical price",
    line=dict(color="#8B4513", width=1.5)
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast["ds"],
    y=forecast["yhat"],
    name="Price forecast",
    line=dict(color="#1D9E75", width=2, dash="dot")
))

# Confidence interval
fig.add_trace(go.Scatter(
    x=list(forecast["ds"]) + list(forecast["ds"][::-1]),
    y=list(forecast["yhat_upper"]) + list(forecast["yhat_lower"][::-1]),
    fill="toself",
    fillcolor="rgba(29,158,117,0.1)",
    line=dict(color="rgba(255,255,255,0)"),
    name="95% confidence interval"
))

fig.update_layout(
    title="Arabica Coffee - Price Forecast (12 months ahead)",
    xaxis_title="Date",
    yaxis_title="Price (USD/kg)",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(range=["2015-01-01", "2027-06-01"])
)